# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shrishagk/My_flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
from getpass import getpass
import duckdb

HF_TOKEN = getpass("Enter your Hugging Face token: ")

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

In [6]:

REL = "hf://datasets/FlyRank/internship-warehouse"

PERF_JAN = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2025-01/*.parquet'"
    f")"
)

print(con.sql(f"SELECT * FROM {PERF_JAN} LIMIT 5").df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
3  2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
4  2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   
3            True            True                True               False   
4            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...     

In [7]:

jan_sample = con.sql(f'SELECT * FROM {PERF_JAN}').df()

print(jan_sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [13]:
import numpy as np
import pandas as pd

df = jan_sample.copy()

df["report_date"] = pd.to_datetime(df["report_date"])

# GSC CTR
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    np.nan
)

# Organic-session share
df["total_sessions"] = (
    df["sessions_organic"].fillna(0)
    + df["sessions_direct"].fillna(0)
    + df["sessions_referral"].fillna(0)
    + df["sessions_social"].fillna(0)
    + df["sessions_paid"].fillna(0)
    + df["sessions_ai"].fillna(0)
)

df["organic_share"] = np.where(
    df["total_sessions"] > 0,
    df["sessions_organic"] / df["total_sessions"],
    np.nan
)

print("Rows:", len(df))
print("Date range:", df["report_date"].min().date(), "to", df["report_date"].max().date())

for col in ["ctr", "gsc_avg_position", "organic_share"]:
    print(f"\n--- {col} ---")
    print(df[col].describe())
    print("Missing:", df[col].isna().sum())
    print("P90:", df[col].quantile(0.90))
    print("P99:", df[col].quantile(0.99))

Rows: 1297
Date range: 2025-01-27 to 2025-01-31

--- ctr ---
count    1297.000000
mean        0.007432
std         0.043044
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.500000
Name: ctr, dtype: float64
Missing: 0
P90: 0.0
P99: 0.25

--- gsc_avg_position ---
count    1297.000000
mean       35.465284
std        28.744245
min         0.000000
25%         9.333333
50%        27.500000
75%        58.000000
max       101.000000
Name: gsc_avg_position, dtype: float64
Missing: 0
P90: 81.80000000000004
P99: 97.51999999999998

--- organic_share ---
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: organic_share, dtype: float64
Missing: 1297
P90: nan
P99: nan


In [14]:
first_date = df["report_date"].min()
latest_date = df["report_date"].max()

first_clicks = (
    df[df["report_date"] == first_date]
    [["client_hash_id", "content_hash_id", "gsc_clicks"]]
    .rename(columns={"gsc_clicks": "first_clicks"})
)

latest_clicks = (
    df[df["report_date"] == latest_date]
    [["client_hash_id", "content_hash_id", "gsc_clicks"]]
    .rename(columns={"gsc_clicks": "latest_clicks"})
)

trend = first_clicks.merge(
    latest_clicks,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

trend["click_change_pct"] = np.where(
    trend["first_clicks"] > 0,
    (trend["latest_clicks"] - trend["first_clicks"])
    / trend["first_clicks"],
    np.nan
)

trend["trend_down"] = trend["click_change_pct"] < 0

print("\n--- Short-window trend ---")
print("Content items with both dates:", len(trend))
print("Observed declines:", trend["trend_down"].sum())
print(
    "Decline rate among items with measurable first clicks:",
    trend.loc[trend["first_clicks"] > 0, "trend_down"].mean()
)


--- Short-window trend ---
Content items with both dates: 160
Observed declines: 8
Decline rate among items with measurable first clicks: 0.7272727272727273


The main baseline signals are short-window click change, GSC CTR, and average search position. The baseline also includes organic-session share when measurable, but the January sample does not provide a usable threshold for that signal. I will inspect the distributions and tails before deciding whether these signals support the baseline assumptions.


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

I will test three signals used by the baseline: short-window click decline, CTR, and average search position. The tests focus on whether the observed distributions support the assumptions behind the rule, rather than treating association as proof of causation.


In [16]:
# Signal test #1: short-window decline

print("First date:", first_date.date())
print("Latest date:", latest_date.date())

print("\nItems with measurable first clicks:")
print((trend["first_clicks"] > 0).sum())

print("\nObserved declines:")
print(trend["trend_down"].sum())

print("\nObserved increases:")
print(
    (
        (trend["first_clicks"] > 0)
        & (trend["click_change_pct"] > 0)
    ).sum()
)

print("\nNo measurable percentage change:")
print(trend["click_change_pct"].isna().sum())

First date: 2025-01-27
Latest date: 2025-01-31

Items with measurable first clicks:
11

Observed declines:
8

Observed increases:
1

No measurable percentage change:
149


In [17]:
# Signal test #2: CTR

ctr = df["ctr"].dropna()

print("CTR observations:", len(ctr))
print("CTR = 0:", (ctr == 0).sum())
print("CTR > 0:", (ctr > 0).sum())

print("\nCTR distribution:")
print(ctr.describe())

print("\nCTR quartiles:")
print(ctr.quantile([0.25, 0.50, 0.75, 0.90, 0.99]))

CTR observations: 1297
CTR = 0: 1220
CTR > 0: 77

CTR distribution:
count    1297.000000
mean        0.007432
std         0.043044
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         0.500000
Name: ctr, dtype: float64

CTR quartiles:
0.25    0.00
0.50    0.00
0.75    0.00
0.90    0.00
0.99    0.25
Name: ctr, dtype: float64


In [18]:
# Signal test #3: average search position

position = df["gsc_avg_position"].dropna()

position_threshold = position.quantile(0.75)

print("Position observations:", len(position))
print("Median:", position.median())
print("75th percentile:", position_threshold)
print("90th percentile:", position.quantile(0.90))
print("99th percentile:", position.quantile(0.99))

print(
    "\nRows at or worse than 75th percentile:",
    (position >= position_threshold).sum()
)

Position observations: 1297
Median: 27.5
75th percentile: 58.0
90th percentile: 81.80000000000004
99th percentile: 97.51999999999998

Rows at or worse than 75th percentile: 329


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The baseline gives the largest weight to TREND_DOWN, so this audit tests the assumption behind that flag. The assumption is that a decline from the earliest to latest available observation is useful evidence for prioritizing a content item. Because the January sample covers only five daily observations, I will check how often the decline is measurable and how often it occurs among items with non-zero starting clicks.


In [19]:
# Flag-linked test: is TREND_DOWN sufficiently measurable?

measurable = trend[trend["first_clicks"] > 0].copy()

print("Total content items:", len(trend))
print("Items with first-day clicks > 0:", len(measurable))

print(
    "Items showing observed decline:",
    measurable["trend_down"].sum()
)

print(
    "Items showing observed increase:",
    (measurable["click_change_pct"] > 0).sum()
)

print(
    "Items with no percentage change because first clicks were zero:",
    (trend["first_clicks"] == 0).sum()
)

print(
    "\nDecline rate among measurable items:",
    round(measurable["trend_down"].mean(), 3)
)

Total content items: 160
Items with first-day clicks > 0: 11
Items showing observed decline: 8
Items showing observed increase: 1
Items with no percentage change because first clicks were zero: 149

Decline rate among measurable items: 0.727


**Verdict: MIXED.** The data supports using TREND_DOWN as a directional review signal, but the evidence is limited by the five-day observation window. A large number of items have zero clicks at the starting point, making percentage change unavailable, so TREND_DOWN cannot be interpreted as a complete measure of content decline. The flag is therefore useful for prioritization but should not be treated as proof of sustained deterioration.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit suggests that the baseline signals are useful as directional evidence, but they do not independently establish that a page needs refreshing. TREND_DOWN is limited by the short five-day window, while the CTR threshold of zero mainly identifies pages with no observed clicks. A content team should therefore use the combined score to prioritize human review and validate the underlying pattern with a longer performance window before taking action.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.